<a href="https://colab.research.google.com/github/leonardeugenia-hash/TGF2026/blob/main/datos2_contextuales.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)

---
## 1. HÁBITOS DE ESCUCHA
Fuente: Encuesta de Hábitos y Prácticas Culturales — Ministerio de Cultura  
Variable clave: `pct_escucha_diaria` → % TOTAL de españoles que escuchan música todos los días

In [3]:
def leer_habito_escucha(path, year_label):
    """
    Lee un archivo de hábitos de escucha del Ministerio de Cultura.
    Extrae el % TOTAL que escucha música todos los días.
    Devuelve el año central del periodo y el valor.
    """
    df = pd.read_excel(path, sheet_name='tabla-0', header=None, engine='openpyxl')
    # Buscamos la fila TOTAL (la primera fila de datos reales)
    # La columna 0 tiene las etiquetas de fila, la columna 1 es 'Suele escuchar todos los días'
    # Primero identificamos qué fila contiene 'TOTAL'
    mask_total = df[0].astype(str).str.strip().str.upper() == 'TOTAL'
    fila_total = df[mask_total]

    if fila_total.empty:
        print(f"  ⚠️  No se encontró fila TOTAL en {path}")
        return None

    pct = float(fila_total.iloc[0, 1])  # columna 1 = 'Suele escuchar todos los días'
    print(f"  ✅ {year_label}: pct_escucha_diaria TOTAL = {pct}%")
    return pct


# Leemos los 3 ficheros disponibles
print("=" * 50)
print("HÁBITOS DE ESCUCHA")
print("=" * 50)

habitos_raw = {
    2010: leer_habito_escucha('2010_2011_habito_escucha_culturagob.xlsx', '2010-2011'),
    2014: leer_habito_escucha('2014_2015_habito_escucha_culturagob.xlsx', '2014-2015'),
    2018: leer_habito_escucha('2018_2019_habito_escucha_culturagob.xlsx', '2018-2019'),
    2021: leer_habito_escucha('2021_2022_habito_escucha_culturagob.xlsx', '2021-2022'),
}

# Construimos serie temporal y asignamos el año central de cada encuesta
# 2010-2011 → asignamos a 2010 y 2011
# 2014-2015 → asignamos a 2014 y 2015
# 2018-2019 → asignamos a 2018 y 2019
# 2021-2022 → asignamos a 2021 y 2022
habitos_puntos = {
    2010: habitos_raw[2010],
    2011: habitos_raw[2010],
    2014: habitos_raw[2014],
    2015: habitos_raw[2014],
    2018: habitos_raw[2018],
    2019: habitos_raw[2018],
    2021: habitos_raw[2021],
    2022: habitos_raw[2021],
}

df_habitos = pd.DataFrame(list(habitos_puntos.items()), columns=['year', 'pct_escucha_diaria'])
df_habitos = df_habitos.sort_values('year').reset_index(drop=True)

# Interpolamos linealmente los años que faltan (2012, 2013, 2016, 2017, 2020, 2023, 2024)
years_completos = pd.DataFrame({'year': range(2010, 2025)})
df_habitos = years_completos.merge(df_habitos, on='year', how='left')
df_habitos['pct_escucha_diaria'] = df_habitos['pct_escucha_diaria'].interpolate(method='linear').round(2)

print("\nSerie interpolada:")
print(df_habitos.to_string(index=False))

HÁBITOS DE ESCUCHA
  ✅ 2010-2011: pct_escucha_diaria TOTAL = 64.8%
  ✅ 2014-2015: pct_escucha_diaria TOTAL = 65.4%
  ✅ 2018-2019: pct_escucha_diaria TOTAL = 70.6%
  ✅ 2021-2022: pct_escucha_diaria TOTAL = 64.7%

Serie interpolada:
 year  pct_escucha_diaria
 2010               64.80
 2011               64.80
 2012               65.00
 2013               65.20
 2014               65.40
 2015               65.40
 2016               67.13
 2017               68.87
 2018               70.60
 2019               70.60
 2020               67.65
 2021               64.70
 2022               64.70
 2023               64.70
 2024               64.70


/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


---
## 2. SUSCRIPCIONES A PLATAFORMAS DIGITALES DE MÚSICA
Fuente: Encuesta de Hábitos y Prácticas Culturales — Ministerio de Cultura  
Variables clave: `pct_suscripcion_total`, `pct_suscripcion_pago`

In [7]:
def leer_suscripcion(path, year_label):
    """
    Lee un archivo de suscripciones del Ministerio de Cultura.
    Extrae el % TOTAL suscrito y % suscripciones de pago.
    Nota: el fichero 2021-2022 solo tiene datos por tramos de edad, no TOTAL.
    """
    df = pd.read_excel(path, sheet_name='tabla-0', header=None, engine='openpyxl')

    mask_total = df[0].astype(str).str.strip().str.upper() == 'TOTAL'
    fila_total = df[mask_total]

    if fila_total.empty:
        # 2021-2022 no tiene fila TOTAL, estimamos como media de los tramos 15-34
        # (los datos que sí están disponibles)
        mask_15_19 = df[0].astype(str).str.contains('15 a 19', na=False)
        mask_20_24 = df[0].astype(str).str.contains('20 a 24', na=False)
        mask_25_34 = df[0].astype(str).str.contains('25 a 34', na=False)
        filas = df[mask_15_19 | mask_20_24 | mask_25_34]
        if filas.empty:
            print(f"  ⚠️  No se pudo extraer dato de {path}")
            return None, None
        # Tomamos la media de los tramos disponibles como estimación
        total = filas.iloc[:, 1].astype(float).mean().round(1)
        pago  = filas.iloc[:, 3].astype(float).mean().round(1)
        print(f"  ⚠️  {year_label}: sin TOTAL — estimado como media tramos 15-34: total={total}%, pago={pago}%")
        return total, pago

    total = float(fila_total.iloc[0, 1])
    pago  = float(fila_total.iloc[0, 3])
    print(f"  ✅ {year_label}: suscripcion_total={total}%, suscripcion_pago={pago}%")
    return total, pago


print("=" * 50)
print("SUSCRIPCIONES A PLATAFORMAS")
print("=" * 50)

suscripcion_raw = {}
t, p = leer_suscripcion('2018_2019_suscripcion_culturagob.xlsx', '2018-2019')
suscripcion_raw[2018] = (t, p)
suscripcion_raw[2019] = (t, p)

t, p = leer_suscripcion('2021_2022_suscripcion_culturagob.xlsx', '2021-2022')
suscripcion_raw[2021] = (t, p)
suscripcion_raw[2022] = (t, p)

# Antes de 2018 no existe la encuesta de suscripciones → imputamos 0 en 2010
# (Spotify llegó a España en 2013; suscripciones eran prácticamente 0 antes)
suscripcion_raw[2010] = (0.0, 0.0)
suscripcion_raw[2013] = (2.0, 1.0)  # estimación conservadora: inicio Spotify España

df_suscripcion = pd.DataFrame(
    [(y, v[0], v[1]) for y, v in sorted(suscripcion_raw.items())],
    columns=['year', 'pct_suscripcion_total', 'pct_suscripcion_pago']
)

years_completos = pd.DataFrame({'year': range(2010, 2025)})
df_suscripcion = years_completos.merge(df_suscripcion, on='year', how='left')
df_suscripcion['pct_suscripcion_total'] = df_suscripcion['pct_suscripcion_total'].interpolate(method='linear').round(2)
df_suscripcion['pct_suscripcion_pago']  = df_suscripcion['pct_suscripcion_pago'].interpolate(method='linear').round(2)

print("\nSerie interpolada:")
print(df_suscripcion.to_string(index=False))

SUSCRIPCIONES A PLATAFORMAS
  ✅ 2018-2019: suscripcion_total=21.4%, suscripcion_pago=10.7%
  ⚠️  2021-2022: sin TOTAL — estimado como media tramos 15-34: total=55.9%, pago=29.2%

Serie interpolada:
 year  pct_suscripcion_total  pct_suscripcion_pago
 2010                   0.00                  0.00
 2011                   0.67                  0.33
 2012                   1.33                  0.67
 2013                   2.00                  1.00
 2014                   5.88                  2.94
 2015                   9.76                  4.88
 2016                  13.64                  6.82
 2017                  17.52                  8.76
 2018                  21.40                 10.70
 2019                  21.40                 10.70
 2020                  38.65                 19.95
 2021                  55.90                 29.20
 2022                  55.90                 29.20
 2023                  55.90                 29.20
 2024                  55.90         

/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


---
## 3. CONCIERTOS DE MÚSICA ACTUAL
Fuente: Encuesta de Hábitos y Prácticas Culturales — Ministerio de Cultura  
Variable clave: `asistentes_conciertos_miles` → total de personas que fueron a conciertos de música actual

In [8]:
def leer_conciertos(path, year_label):
    """
    Lee el archivo de conciertos del Ministerio de Cultura.
    Extrae el total de asistentes a conciertos de música actual (en miles).
    """
    df = pd.read_excel(path, sheet_name='tabla-0', header=None, engine='openpyxl')

    mask_total = df[0].astype(str).str.strip().str.upper() == 'TOTAL'
    fila_total = df[mask_total]

    if fila_total.empty:
        print(f"  ⚠️  No se encontró fila TOTAL en {path}")
        return None

    # Columna 7 = 'Música actual: Total han ido en un año (Miles)'
    # (mirando la estructura del fichero 2018-2019)
    valor = float(fila_total.iloc[0, 7])
    print(f"  ✅ {year_label}: asistentes_conciertos_actuales = {valor} miles")
    return valor


print("=" * 50)
print("CONCIERTOS DE MÚSICA ACTUAL")
print("=" * 50)

conciertos_raw = {}
v = leer_conciertos('2018_2019_conciertos_culturagob.xlsx', '2018-2019')
conciertos_raw[2018] = v
conciertos_raw[2019] = v

# Nota: el fichero 2021-2022 es .xls y requiere xlrd que no está disponible.
# Si en el futuro lo conviertes a .xlsx, usa: leer_conciertos('2021_2022_conciertos_culturagob.xlsx', '2021-2022')
# Por ahora lo dejamos como NaN — se interpolará
print("  ⚠️  2021-2022: archivo .xls no legible sin xlrd — se interpolará")

df_conciertos = pd.DataFrame(
    [(y, v) for y, v in sorted(conciertos_raw.items())],
    columns=['year', 'asistentes_conciertos_miles']
)

years_completos = pd.DataFrame({'year': range(2010, 2025)})
df_conciertos = years_completos.merge(df_conciertos, on='year', how='left')
# Con solo un punto de datos reales, la interpolación no tiene suficiente base;
# usamos forward fill + backward fill como estimación conservadora
df_conciertos['asistentes_conciertos_miles'] = (
    df_conciertos['asistentes_conciertos_miles']
    .interpolate(method='linear')
    .fillna(method='bfill')
    .fillna(method='ffill')
    .round(0)
)

print("\nSerie (con interpolación):")
print(df_conciertos.to_string(index=False))
print("\n→ Nota: datos de conciertos limitados a 2018-2019. Interpreta esta variable con cautela.")

CONCIERTOS DE MÚSICA ACTUAL
  ✅ 2018-2019: asistentes_conciertos_actuales = 46.9 miles
  ⚠️  2021-2022: archivo .xls no legible sin xlrd — se interpolará

Serie (con interpolación):
 year  asistentes_conciertos_miles
 2010                         47.0
 2011                         47.0
 2012                         47.0
 2013                         47.0
 2014                         47.0
 2015                         47.0
 2016                         47.0
 2017                         47.0
 2018                         47.0
 2019                         47.0
 2020                         47.0
 2021                         47.0
 2022                         47.0
 2023                         47.0
 2024                         47.0

→ Nota: datos de conciertos limitados a 2018-2019. Interpreta esta variable con cautela.


/tmp/ipykernel_19862/3845861833.py:48: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  .fillna(method='bfill')
/tmp/ipykernel_19862/3845861833.py:49: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  .fillna(method='ffill')


---
## 4. GOOGLE TRENDS
Fuentes: `webtrends_spotifypremium_2010_2024.csv` y `webtrends_streamingmusic_2010_2024.csv`  
Variables: índice de búsqueda mensual → agregamos a nivel anual (media)

In [9]:
def leer_google_trends(path, nombre_col):
    """
    Lee un CSV de Google Trends exportado con cabecera de categoría en fila 0.
    Devuelve un DataFrame anual con la media del índice de búsqueda.
    """
    # Fila 0 es la categoría, fila 1 es la cabecera real
    df = pd.read_csv(path, skiprows=1, encoding='utf-8')
    df.columns = ['month', 'trend_index']

    # Convertimos mes a año
    df['month'] = pd.to_datetime(df['month'], format='%Y-%m')
    df['year'] = df['month'].dt.year

    # Los valores <NA> o 0 de Google Trends son semanas sin datos, los tratamos
    df['trend_index'] = pd.to_numeric(df['trend_index'], errors='coerce')

    # Agregamos por año: media ignorando NaN
    df_anual = df.groupby('year')['trend_index'].mean().round(1).reset_index()
    df_anual.columns = ['year', nombre_col]

    return df_anual


print("=" * 50)
print("GOOGLE TRENDS")
print("=" * 50)

df_trends_premium  = leer_google_trends('webtrends_spotifypremium_2010_2024.csv',   'trend_spotify_premium')
df_trends_streaming = leer_google_trends('webtrends_streamingmusic_2010_2024.csv', 'trend_streaming_music')

# Filtramos al rango 2010-2024
df_trends_premium   = df_trends_premium[(df_trends_premium['year'] >= 2010) & (df_trends_premium['year'] <= 2024)]
df_trends_streaming = df_trends_streaming[(df_trends_streaming['year'] >= 2010) & (df_trends_streaming['year'] <= 2024)]

df_trends = df_trends_premium.merge(df_trends_streaming, on='year', how='outer').sort_values('year').reset_index(drop=True)

print("\nNulos en trends:", df_trends.isnull().sum().to_dict())
df_trends = df_trends.fillna(df_trends.mean(numeric_only=True).round(1))

print("\nSerie anual:")
print(df_trends.to_string(index=False))

GOOGLE TRENDS


FileNotFoundError: [Errno 2] No such file or directory: 'webtrends_spotifypremium_2010_2024.csv'

---
## 5. VALIDACIÓN INDIVIDUAL DE CADA DATASET

In [ ]:
print("=" * 55)
print("VALIDACIÓN")
print("=" * 55)

datasets = {
    'df_habitos':     df_habitos,
    'df_suscripcion': df_suscripcion,
    'df_conciertos':  df_conciertos,
    'df_trends':      df_trends,
}

for name, df in datasets.items():
    print(f"\n{name}")
    print(f"  Shape: {df.shape}")
    print(f"  Años:  {df['year'].min()} – {df['year'].max()}")
    nulos = df.isnull().sum()
    print(f"  Nulos: {nulos[nulos > 0].to_dict() if nulos.sum() > 0 else 'ninguno'}")

---
## 6. INTEGRACIÓN: MERGE DE TODOS LOS DATASETS EN UNO

In [ ]:
print("=" * 55)
print("MERGE FINAL")
print("=" * 55)

df_contextual = (
    df_habitos
    .merge(df_suscripcion, on='year', how='left')
    .merge(df_conciertos,  on='year', how='left')
    .merge(df_trends,      on='year', how='left')
)

print(f"Shape final: {df_contextual.shape}")
print(f"Columnas: {df_contextual.columns.tolist()}")
print("\nNulos:")
print(df_contextual.isnull().sum())
print("\nDataset completo:")
print(df_contextual.to_string(index=False))

---
## 7. MERGE CON EL DATASET DE SPOTIFY
Unimos con `dataset_spotify_final_year.csv` del Script 1

In [ ]:
print("=" * 55)
print("MERGE CON SPOTIFY")
print("=" * 55)

df_spotify = pd.read_csv('dataset_spotify_final_year.csv')
df_spotify['year'] = df_spotify['year'].astype(int)

print(f"Spotify shape: {df_spotify.shape}")
print(f"Columnas Spotify: {df_spotify.columns.tolist()}")

df_final = df_spotify.merge(df_contextual, on='year', how='left')

print(f"\nDataset final shape: {df_final.shape}")
print("\nNulos en dataset final:")
print(df_final.isnull().sum())
print("\nVista previa:")
df_final.head()

---
## 8. VISUALIZACIONES DE LAS VARIABLES CONTEXTUALES

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
fig.suptitle('Evolución de variables contextuales en España (2010-2024)', fontsize=14, fontweight='bold')

vars_plot = [
    ('pct_escucha_diaria',       'Hábito escucha diaria (%)',       axes[0, 0]),
    ('pct_suscripcion_total',    'Suscripción plataformas (%)',      axes[0, 1]),
    ('pct_suscripcion_pago',     'Suscripción de pago (%)',         axes[0, 2]),
    ('asistentes_conciertos_miles', 'Asistentes conciertos (miles)', axes[1, 0]),
    ('trend_spotify_premium',    'Google Trends: Spotify Premium',  axes[1, 1]),
    ('trend_streaming_music',    'Google Trends: Streaming Music',  axes[1, 2]),
]

for col, title, ax in vars_plot:
    if col in df_final.columns:
        ax.plot(df_final['year'], df_final[col], marker='o', linewidth=2, color='steelblue')
        ax.set_title(title, fontsize=11)
        ax.set_xlabel('Año')
        ax.grid(True, alpha=0.3)
    else:
        ax.set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# Correlación entre variables contextuales y popularidad musical
cols_corr = [
    'avg_popularity', 'avg_duration',
    'pct_escucha_diaria', 'pct_suscripcion_total', 'pct_suscripcion_pago',
    'trend_spotify_premium', 'trend_streaming_music',
    'pib_per_capita', 'internet_access'
]

# Solo usamos columnas que existan en el dataset final
cols_corr = [c for c in cols_corr if c in df_final.columns]

plt.figure(figsize=(10, 7))
sns.heatmap(
    df_final[cols_corr].corr(numeric_only=True),
    annot=True, fmt='.2f', cmap='coolwarm', center=0,
    linewidths=0.5
)
plt.title('Correlación: variables musicales × variables contextuales', fontsize=12)
plt.tight_layout()
plt.show()

---
## 9. GUARDADO DEL DATASET FINAL

In [ ]:
print("=" * 55)
print("GUARDADO")
print("=" * 55)

# Dataset contextual (solo variables externas)
df_contextual.to_csv('dataset_contextual_final_year.csv', index=False)
print("✅ dataset_contextual_final_year.csv guardado")

# Dataset completo (Spotify + contexto)
df_final.to_csv('dataset_completo_final_year.csv', index=False)
print("✅ dataset_completo_final_year.csv guardado")

print(f"\nResumen final:")
print(f"  Filas: {df_final.shape[0]}")
print(f"  Columnas: {df_final.shape[1]}")
print(f"  Años: {df_final['year'].min()} – {df_final['year'].max()}")
print(f"  Nulos totales: {df_final.isnull().sum().sum()}")
print(f"\nColumnas del dataset final:")
for col in df_final.columns:
    print(f"  - {col}")